# 02 - Compreensão e Avaliação da Qualidade dos Dados

Este notebook carrega os 9 arquivos de `data/raw/` (gerados em `01_ingestao.ipynb`) e investiga, para cada um,
os problemas de qualidade relevantes: valores ausentes, registros duplicados, tipos de dados incorretos,
formatos inconsistentes, valores inválidos, categorias inconsistentes e outliers.

Os achados aqui orientam diretamente as decisões de limpeza em `03_limpeza_preparacao.ipynb`.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

RAW = "../data/raw/"

orders = pd.read_csv(RAW + "olist_orders_dataset.csv")
customers = pd.read_csv(RAW + "olist_customers_dataset.csv")
items = pd.read_csv(RAW + "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW + "olist_products_dataset.csv")
sellers = pd.read_csv(RAW + "olist_sellers_dataset.csv")
geo = pd.read_csv(RAW + "olist_geolocation_dataset.csv")
cat_translation = pd.read_csv(RAW + "product_category_name_translation.csv")

tabelas = {
    "orders": orders, "customers": customers, "items": items, "payments": payments,
    "reviews": reviews, "products": products, "sellers": sellers, "geo": geo,
    "cat_translation": cat_translation,
}

## Visão geral: formato, tipos e valores ausentes por tabela

In [2]:
for nome, df in tabelas.items():
    print(f"=== {nome} - {df.shape[0]} linhas x {df.shape[1]} colunas ===")
    print(df.dtypes.to_string())
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos):
        print("\nValores ausentes:")
        print(nulos.to_string())
    print()

=== orders - 99441 linhas x 8 colunas ===
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

Valores ausentes:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

=== customers - 99441 linhas x 5 colunas ===
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str

=== items - 112650 linhas x 7 colunas ===
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

=== payments - 103886 linhas x 5 colunas ===
order_i

**Achado 1 - tipos de dados incorretos:** todas as colunas de data/hora (`orders`, `items`, `reviews`) estão
como texto (`object`), não `datetime`. Isso impede cálculos de prazo de entrega e precisa ser corrigido na limpeza.

**Achado 2 - valores ausentes legítimos:**
- `orders`: `order_approved_at` (160), `order_delivered_carrier_date` (1.783) e `order_delivered_customer_date`
  (2.965) ficam nulos quando o pedido nunca chegou a essa etapa (cancelado, indisponível, ainda em processamento).
  Verificamos isso abaixo cruzando com `order_status`.
- `reviews`: `review_comment_title` e `review_comment_message` têm milhares de nulos porque são campos de texto
  opcionais - o cliente pode avaliar sem escrever comentário.
- `products`: ~610 produtos sem `product_category_name` e metadados associados, e 2 produtos sem dimensões físicas.
  São produtos cadastrados de forma incompleta pelos vendedores.

In [3]:
# Confirma que os nulos de data em 'orders' se concentram em pedidos não concluídos
orders.groupby("order_status")[["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]].apply(
    lambda g: g.isnull().sum()
)

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


## Registros duplicados

In [4]:
for nome, df in tabelas.items():
    print(nome, "- duplicatas de linha inteira:", df.duplicated().sum())

orders - duplicatas de linha inteira: 0
customers - duplicatas de linha inteira: 0
items - duplicatas de linha inteira: 0


payments - duplicatas de linha inteira: 0
reviews - duplicatas de linha inteira: 0
products - duplicatas de linha inteira: 0
sellers - duplicatas de linha inteira: 0


geo - duplicatas de linha inteira: 261831
cat_translation - duplicatas de linha inteira: 0


In [5]:
# geo concentra quase todas as duplicatas do dataset: investigar
dup_geo = geo.duplicated().sum()
print(f"geolocation: {dup_geo} linhas duplicadas de {len(geo)} ({dup_geo/len(geo):.1%})")
geo[geo.duplicated(keep=False)].sort_values("geolocation_zip_code_prefix").head(6)

geolocation: 261831 linhas duplicadas de 1000163 (26.2%)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
851,1001,-23.549825,-46.633970,sao paulo,SP
1062,1001,-23.550498,-46.634338,sao paulo,SP
1246,1001,-23.549292,-46.633559,sao paulo,SP
897,1001,-23.549292,-46.633559,sao paulo,SP
99,1001,-23.549292,-46.633559,sao paulo,SP
864,1001,-23.549825,-46.633970,sao paulo,SP


**Achado 3 - duplicatas relevantes:** `olist_geolocation_dataset.csv` tem ~26% de linhas 100% duplicadas
(mesmo CEP, mesma lat/lng, mesma cidade/estado). Isso infla o arquivo sem adicionar nenhuma informação nova e deve
ser removido na limpeza.

As demais tabelas não têm duplicatas de linha inteira.

## Chaves que deveriam ser únicas

In [6]:
print("order_id duplicado em orders:", orders["order_id"].duplicated().sum())
print("review_id duplicado em reviews:", reviews["review_id"].duplicated().sum())
print("review_id + order_id duplicado:", reviews.duplicated(subset=["review_id", "order_id"]).sum())

exemplo = reviews[reviews["review_id"].duplicated(keep=False)].sort_values("review_id")
exemplo[["review_id", "order_id", "review_score"]].head(4)

order_id duplicado em orders: 0
review_id duplicado em reviews: 814
review_id + order_id duplicado: 0


,review_id,order_id,review_score
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5


**Achado 4 - `review_id` não é uma chave única:** 814 valores de `review_id` aparecem em mais de um `order_id`
diferente - é uma particularidade conhecida do dataset da Olist (o mesmo texto de avaliação pode ter sido
reaproveitado entre pedidos). Como o par `(review_id, order_id)` **é** único, a granularidade correta de análise
é o pedido, não a avaliação isolada. Não é um erro a corrigir, é uma característica a documentar.

## Valores inválidos ou impossíveis

In [7]:
print("=== items: price e freight_value ===")
print(items[["price", "freight_value"]].describe())
print("price <= 0:", (items["price"] <= 0).sum(), "| freight_value < 0:", (items["freight_value"] < 0).sum())

print("\n=== payments ===")
print(payments[["payment_value", "payment_installments"]].describe())
print("payment_value <= 0:", (payments["payment_value"] <= 0).sum())
print("payment_installments == 0:", (payments["payment_installments"] == 0).sum())
print(payments["payment_type"].value_counts())

=== items: price e freight_value ===
               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000
price <= 0: 0 | freight_value < 0: 0

=== payments ===
       payment_value  payment_installments
count  103886.000000         103886.000000
mean      154.100380              2.853349
std       217.494064              2.687051
min         0.000000              0.000000
25%        56.790000              1.000000
50%       100.000000              1.000000
75%       171.837500              4.000000
max     13664.080000             24.000000
payment_value <= 0: 9
payment_installments == 0: 2
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [8]:
payments[payments["payment_value"] <= 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


**Achado 5 - pagamentos inválidos:**
- 3 linhas têm `payment_type == "not_defined"` **e** `payment_value == 0`: não representam nenhuma forma de
  pagamento real, parecem registros de placeholder - candidatas à remoção.
- As outras 6 linhas com `payment_value == 0` são `payment_type == "voucher"`: fazem parte de pagamentos
  fracionados em múltiplas parcelas de voucher (`payment_sequential` > 1) e são legítimas - não remover.
- 2 linhas têm `payment_installments == 0` com `payment_type == "credit_card"` e valor > 0: fisicamente
  impossível (mesmo um pagamento à vista conta como 1 parcela neste esquema) - valor inválido a corrigir.

In [9]:
print("=== products: dimensões físicas ===")
print(products[["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]].describe())
print("product_weight_g == 0:", (products["product_weight_g"] == 0).sum())

=== products: dimensões físicas ===
       product_weight_g  product_length_cm  product_height_cm  \
count      32949.000000       32949.000000       32949.000000   
mean        2276.472488          30.815078          16.937661   
std         4282.038731          16.914458          13.637554   
min            0.000000           7.000000           2.000000   
25%          300.000000          18.000000           8.000000   
50%          700.000000          25.000000          13.000000   
75%         1900.000000          38.000000          21.000000   
max        40425.000000         105.000000         105.000000   

       product_width_cm  
count      32949.000000  
mean          23.196728  
std           12.079047  
min            6.000000  
25%           15.000000  
50%           20.000000  
75%           30.000000  
max          118.000000  
product_weight_g == 0: 4


**Achado 6 - peso zero:** 4 produtos têm `product_weight_g == 0`, o que é fisicamente impossível para um item
físico vendido - deve ser tratado como ausente, não como zero real.

## Categorias inconsistentes

In [10]:
estados_validos = {
    "AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT","PA","PB","PE","PI",
    "PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO",
}
print("estados inválidos em customers:", (~customers["customer_state"].isin(estados_validos)).sum())
print("estados inválidos em sellers:", (~sellers["seller_state"].isin(estados_validos)).sum())
print("estados inválidos em geo:", (~geo["geolocation_state"].isin(estados_validos)).sum())

cats_traduzidas = set(cat_translation["product_category_name"])
cats_produtos = set(products["product_category_name"].dropna())
print("\ncategorias de produto sem tradução em inglês:", cats_produtos - cats_traduzidas)

estados inválidos em customers: 0
estados inválidos em sellers: 0
estados inválidos em geo: 0

categorias de produto sem tradução em inglês: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}


In [11]:
import re

padrao_valido = re.compile(r"^[a-z0-9áéíóúãõâêôçàü\s\-\.\(\)/']*$")

def contar_nomes_invalidos(df, coluna):
    mask = ~df[coluna].fillna("").str.match(padrao_valido)
    return mask

for nome_df, df, coluna in [("customers", customers, "customer_city"), ("sellers", sellers, "seller_city"), ("geo", geo, "geolocation_city")]:
    mask = contar_nomes_invalidos(df, coluna)
    print(f"{nome_df}.{coluna}: {mask.sum()} valores com caracteres fora do esperado (de {len(df)})")

mask_sigla = geo["geolocation_city"].str.lower().isin({s.lower() for s in estados_validos})
print("\ngeo.geolocation_city igual a uma sigla de estado (inválido):", mask_sigla.sum())
print(geo.loc[mask_sigla, "geolocation_city"].value_counts())

customers.customer_city: 0 valores com caracteres fora do esperado (de 99441)
sellers.seller_city: 7 valores com caracteres fora do esperado (de 3095)


geo.geolocation_city: 12 valores com caracteres fora do esperado (de 1000163)

geo.geolocation_city igual a uma sigla de estado (inválido): 4
geolocation_city
sp    3
rj    1
Name: count, dtype: int64


**Achado 7 - categorias/texto inconsistente:**
- Estados (`customer_state`, `seller_state`, `geolocation_state`) estão todos dentro das 27 UFs válidas - sem
  problema aqui.
- Duas categorias de produto (`pc_gamer` e `portateis_cozinha_e_preparadores_de_alimentos`) não têm entrada na
  tabela de tradução - ficarão sem `product_category_name_english` após a junção; é uma lacuna pequena e será
  documentada, não inventada.
- `geolocation_city` tem qualidade bem inferior às colunas equivalentes de `customers`/`sellers`: ~16 linhas com
  texto corrompido (entidades HTML como `florian&oacute;polis`, trechos de URL-encoding, pontuação solta) e 4
  linhas em que o valor da cidade é na verdade uma sigla de estado (`sp`, `rj`) - tudo isso é inválido como nome
  de cidade. Além disso, a mesma cidade aparece sob múltiplas grafias (ex.: 22 variantes diferentes contendo
  "paulo"), a maioria apenas por existirem *outras* cidades cujo nome contém "paulo" (ex. "paulo afonso"), não
  por erro - mas há também duplicação de grafia genuína (com/sem acento). Corrigir a canonicalização completa de
  ~8.000 nomes de cidade exigiria uma tabela de referência externa e está fora do escopo do RA1; a decisão é
  nulificar apenas os ~20 valores claramente inválidos e documentar a limitação residual.

## Outliers (coordenadas fora do território brasileiro)

In [12]:
fora_do_brasil = geo[
    (geo["geolocation_lat"] > 6) | (geo["geolocation_lat"] < -34)
    | (geo["geolocation_lng"] > -32) | (geo["geolocation_lng"] < -74)
]
print(f"{len(fora_do_brasil)} de {len(geo)} coordenadas fora da caixa delimitadora aproximada do Brasil")
fora_do_brasil.head()

31 de 1000163 coordenadas fora da caixa delimitadora aproximada do Brasil

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ


**Achado 8 - outliers de geolocalização:** 31 registros têm latitude/longitude fora da caixa delimitadora
aproximada do território brasileiro (lat entre -34 e 6, lng entre -74 e -32) - coordenadas de geocodificação
claramente erradas para um dataset que é 100% de pedidos no Brasil. Serão removidos na limpeza, pois usá-los em
qualquer cálculo de distância no RA2 produziria resultados sem sentido.

## Formato dos identificadores de CEP (zip code prefix)

In [13]:
print("prefixos de CEP < 10000 (perderiam o zero à esquerda se exibidos como string sem padding):")
print("customers:", (customers["customer_zip_code_prefix"] < 10000).sum(), "de", len(customers))
print("sellers:", (sellers["seller_zip_code_prefix"] < 10000).sum(), "de", len(sellers))
print("geo:", (geo["geolocation_zip_code_prefix"] < 10000).sum(), "de", len(geo))

prefixos de CEP < 10000 (perderiam o zero à esquerda se exibidos como string sem padding):
customers: 23995 de 99441
sellers: 1027 de 3095
geo: 245733 de 1000163


**Achado 9 - tipo de dado incorreto (semântico):** os prefixos de CEP estão como número inteiro, mas CEP é um
identificador categórico, não uma quantidade - e boa parte dos prefixos reais começam com `0` (região de São
Paulo), que se perde na representação numérica ao exibir/exportar. Serão convertidos para texto com 5 dígitos
(zero-padded) na limpeza.

## Resumo dos achados

| Tabela | Problema | Ação planejada |
|---|---|---|
| orders | datas como texto | converter para datetime |
| orders | nulos em datas de aprovação/entrega | manter (ausência legítima por status) |
| items | `shipping_limit_date` como texto | converter para datetime |
| payments | 3 linhas `not_defined` + valor 0 | remover |
| payments | 2 linhas com `installments == 0` | corrigir para 1 |
| reviews | datas como texto | converter para datetime |
| reviews | `review_id` duplicado entre pedidos | manter, documentar (não é erro) |
| products | 610 linhas sem categoria/metadados | manter nulo (ausência legítima) |
| products | 4 linhas com peso 0 | converter para nulo (valor impossível) |
| products | 2 categorias sem tradução em inglês | manter nulo após junção, documentar |
| sellers/customers | prefixo de CEP numérico | converter para texto com 5 dígitos |
| geo | ~26% duplicatas de linha inteira | remover |
| geo | 31 coordenadas fora do Brasil | remover |
| geo | ~20 nomes de cidade inválidos/corrompidos | converter para nulo |
| geo | prefixo de CEP numérico | converter para texto com 5 dígitos |

Detalhamento completo em `docs/03_relatorio_qualidade.md`. Próxima etapa: `03_limpeza_preparacao.ipynb`.